# Z-Image Turbo on Kaggle (ComfyUI)

Z-Image Turbo (Tongyi-MAI / Alibaba, 6B S3-DiT, Apache 2.0, commercial OK). ComfyUI native support.
Official template: Comfy-Org/workflow_templates `image_z_image_turbo.json` (subgraph expanded below).
Nodes: UNETLoader -> ModelSamplingAuraFlow (shift 3) -> KSampler (8 steps, cfg 1, res_multistep, simple);
CLIPLoader type lumina2 (qwen_3_4b) -> CLIPTextEncode -> ConditioningZeroOut (negative);
EmptySD3LatentImage; VAEDecode; SaveImage.

Files (sizes confirmed via HF API, Comfy-Org/Z-Image-Turbo split_files/):
- diffusion z_image_turbo_int8_convrot.safetensors 6.20 GB (INT8; BF16 12.31 GB + BF16 encoder 8.04 GB + VAE 0.34 GB = 20.69 GB would exceed the 20 GB /kaggle/working limit)
- text encoder qwen_3_4b_fp8_mixed.safetensors 5.63 GB (FP8; BF16 encoder is 8.04 GB)
- VAE ae.safetensors 0.34 GB (Flux VAE)
Total 12.17 GB, fits /kaggle/working 20 GB. T4 16 GB: FP8/INT8 route runs comfortably.

LoRA (2026-11-15; default all three, user decision 2026-11-15 17:21): three LoraLoaderModelOnly
nodes chained model -> LoRA1 -> LoRA2 -> LoRA3 -> ModelSamplingAuraFlow.
  herraw:   HerRAW_zimage_v0.2_c1-st5000.safetensors, 0.8, dataset kitzhang2019/herraw-zimage-v02
  yaoguang: Kook_Zimage_Yaoguang.safetensors, 1.0, dataset kitzhang2019/kook-zimage-yaoguang2
  ostris:   zimage_turbo_training_adapter_v2.safetensors, 1.0, dataset kitzhang2019/ostris-zimage-turbo-adapter-v2
Each LoRA comes from its own private Kaggle dataset (attached by the App when selected).
Z-Image LoRAs must be trained for Z-Image; FLUX/Qwen LoRAs will NOT work.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!if [ -d /kaggle/working/ComfyUI ]; then echo comfyui-exists; else git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI; fi
!pip install -q -r /kaggle/working/ComfyUI/requirements.txt

# SeedVR2 plugin (numz, 2026-11-15): the SeedVR2VideoUpscaler node upscales the
# low-res generation to 2048x3072 in phase 2. Same install as the verified 7B test.
!if [ -d /kaggle/working/ComfyUI/custom_nodes/ComfyUI-SeedVR2_VideoUpscaler ]; then echo seedvr-plugin-exists; else git clone --depth 1 https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git /kaggle/working/ComfyUI/custom_nodes/ComfyUI-SeedVR2_VideoUpscaler; fi
!cd /kaggle/working/ComfyUI/custom_nodes/ComfyUI-SeedVR2_VideoUpscaler && grep -v -E "^(torch|torchvision)(==|>=|~=|>|<| |$)" requirements.txt > /tmp/seedvr_reqs.txt && pip install -q -r /tmp/seedvr_reqs.txt
!ls /kaggle/working/ComfyUI/custom_nodes/ComfyUI-SeedVR2_VideoUpscaler | head -20


In [ ]:
import os, shutil
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
from huggingface_hub import hf_hub_download
base = "/kaggle/working/ComfyUI/models"
#
# LoRA registry (2026-11-15): three Z-Image LoRAs, each from its own private
# Kaggle dataset (recursive search under /kaggle/input, never hard-code the mount
# path). LORA_NAMES is the per-job selection: patchNotebook overwrites this
# list from the form selection (default: all three, user decision 2026-11-15 17:21;
# empty list = no LoRA).
# LORA_NAME/LORA_STRENGTH keep the first entry for the node-presence check and
# for the single-LoRA fallback in the run cell. LORA_REPO is unused (LoRAs come
# from datasets, not HF) and kept only so older prepare code does not break.
ZIMAGE_LORAS = {
    "herraw": ("HerRAW_zimage_v0.2_c1-st5000.safetensors", 0.8, "kitzhang2019/herraw-zimage-v02"),
    "yaoguang": ("Kook_Zimage_Yaoguang.safetensors", 1.0, "kitzhang2019/kook-zimage-yaoguang2"),
    "ostris": ("zimage_turbo_training_adapter_v2.safetensors", 1.0, "kitzhang2019/ostris-zimage-turbo-adapter-v2"),
}
LORA_NAMES = [ZIMAGE_LORAS[k] for k in ("herraw", "yaoguang", "ostris")]
LORA_NAME = LORA_NAMES[0][0] if LORA_NAMES else ""
LORA_REPO = ""
LORA_STRENGTH = LORA_NAMES[0][1] if LORA_NAMES else 1.0
# Z-Image Turbo files (sizes confirmed via HF API, Comfy-Org/Z-Image-Turbo split_files/):
# - diffusion z_image_turbo_int8_convrot.safetensors 6.20 GB (official Comfy-Org INT8 repack)
# - text encoder qwen_3_4b_fp8_mixed.safetensors 5.63 GB (official Comfy-Org FP8 repack)
# - VAE ae.safetensors 0.34 GB (Flux VAE)
# Total ~12.17 GB, fits /kaggle/working 20 GB.
# NOTE: hf_hub_download mirrors the repo subdirectory (split_files/...) under local_dir,
# so each file is moved to the folder its loader expects (the Wan V3/V4 lesson).
def dl(repo, fname, dest_folder, dest_name):
    p = hf_hub_download(repo_id=repo, filename=fname, local_dir=base)
    target = os.path.join(base, dest_folder, dest_name)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if os.path.abspath(p) != os.path.abspath(target):
        shutil.move(p, target)
    print(dest_folder + "/" + dest_name, round(os.path.getsize(target) / 1e9, 2), "GB")
    return target

dl("Comfy-Org/Z-Image-Turbo", "split_files/diffusion_models/z_image_turbo_int8_convrot.safetensors", "diffusion_models", "z_image_turbo_int8_convrot.safetensors")
dl("Comfy-Org/Z-Image-Turbo", "split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors", "text_encoders", "qwen_3_4b_fp8_mixed.safetensors")
dl("Comfy-Org/Z-Image-Turbo", "split_files/vae/ae.safetensors", "vae", "ae.safetensors")
if LORA_NAMES:
    # Each LoRA comes from its own private dataset, located by recursive
    # search under /kaggle/input and copied into ComfyUI/models/loras/.
    import glob as _lora_glob
    for _lname, _lstr, _lds in LORA_NAMES:
        _hits = sorted(_lora_glob.glob("/kaggle/input/**/" + _lname, recursive=True))
        if not _hits:
            raise SystemExit("lora file not found under /kaggle/input: " + _lname + " (dataset " + _lds + " is not attached to this kernel)")
        _ldest = os.path.join(base, "loras", _lname)
        os.makedirs(os.path.dirname(_ldest), exist_ok=True)
        shutil.copyfile(_hits[0], _ldest)
        print("lora", _lname, "strength", _lstr, round(os.path.getsize(_ldest) / 1e9, 2), "GB", "from", _hits[0])
# Guard: the three files must sit exactly where the loaders look, or fail now.
expected = {
    "diffusion_models": "z_image_turbo_int8_convrot.safetensors",
    "text_encoders": "qwen_3_4b_fp8_mixed.safetensors",
    "vae": "ae.safetensors",
}
for folder, fname in expected.items():
    fp = os.path.join(base, folder, fname)
    print(folder + "/", fname, "exists:", os.path.exists(fp))
    assert os.path.exists(fp), "missing model file: " + fp

# SeedVR2 7B (2026-11-15): DiT seedvr2_ema_7b_fp8_e4m3fn_mixed_block35_fp16.safetensors
# (7.88 GB, AInVFX/SeedVR2_comfyUI) + VAE ema_vae_fp16.safetensors (0.47 GB,
# numz/SeedVR2_comfyUI) come from the private dataset kitzhang2019/seedvr2-7b
# attached to this kernel, located by recursive search under /kaggle/input.
# They are SYMLINKED (not copied) into ComfyUI/models/SEEDVR2/: copying all
# 8.35 GB would overflow /kaggle/working (Qwen already uses ~17 GB of 20 GB).
# Symlinks read through to /kaggle/input, which stays mounted for the run.
import glob as _seedvr_glob
SEEDVR_DIT = "seedvr2_ema_7b_fp8_e4m3fn_mixed_block35_fp16.safetensors"
SEEDVR_VAE = "ema_vae_fp16.safetensors"
seedvr_dir = os.path.join(base, "SEEDVR2")
os.makedirs(seedvr_dir, exist_ok=True)
for _name in (SEEDVR_DIT, SEEDVR_VAE):
    _hits = sorted(_seedvr_glob.glob("/kaggle/input/**/" + _name, recursive=True))
    if not _hits:
        raise SystemExit("seedvr file not found under /kaggle/input: " + _name + " (dataset kitzhang2019/seedvr2-7b is not attached to this kernel)")
    _link = os.path.join(seedvr_dir, _name)
    if os.path.islink(_link) or os.path.exists(_link):
        os.remove(_link)
    os.symlink(_hits[0], _link)
    print("seedvr:", _hits[0], "->", _link, round(os.path.getsize(_link) / 1e9, 2), "GB")
# Guard: the two files must be visible exactly where the loaders look.
for _name in (SEEDVR_DIT, SEEDVR_VAE):
    _fp = os.path.join(seedvr_dir, _name)
    print("SEEDVR2/", _name, "exists:", os.path.exists(_fp))
    assert os.path.exists(_fp), "missing seedvr file: " + _fp
print("seedvr models ready")
print("all models downloaded")


In [ ]:
import json, time, requests, os, subprocess, sys

BASE = "http://127.0.0.1:8188"
LOG = "/kaggle/working/comfy_server.log"

# Start the ComfyUI server as a child of this cell's python process (nohup across cells does not survive Kaggle version runs).
logf = open(LOG, "wb")
proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
    cwd="/kaggle/working/ComfyUI",
    stdout=logf, stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("server pid:", proc.pid)

up = False
for i in range(36):
    if proc.poll() is not None:
        print("server exited early, code:", proc.returncode)
        break
    try:
        r = requests.get(BASE + "/system_stats", timeout=5)
        if r.status_code == 200:
            up = True
            print("server up after", (i + 1) * 5, "s")
            break
    except Exception:
        pass
    time.sleep(5)
if not up:
    print("server did not come up; log tail:")
    print(open(LOG, errors="ignore").read()[-3000:])
    proc.terminate()
    raise SystemExit("comfyui server failed to start")

PROMPT = "A photorealistic portrait of a young woman in a sunlit garden, soft morning light, shallow depth of field, 50mm lens"
WIDTH = 1024
HEIGHT = 1024
# LoRA settings are defined in the download cell; default to "no LoRA" if that cell was skipped.
LORA_NAME = globals().get("LORA_NAME", "")
LORA_STRENGTH = globals().get("LORA_STRENGTH", 1.0)
LORA_NAMES = globals().get("LORA_NAMES", None)

info = requests.get(BASE + "/object_info", timeout=120).json()
have = set(info.keys())
need = ["UNETLoader", "CLIPLoader", "VAELoader", "CLIPTextEncode", "ConditioningZeroOut",
        "EmptySD3LatentImage", "ModelSamplingAuraFlow", "KSampler", "VAEDecode", "SaveImage", "LoadImage", "SeedVR2LoadDiTModel", "SeedVR2LoadVAEModel", "SeedVR2VideoUpscaler"]
if LORA_NAME:
    need.append("LoraLoaderModelOnly")
missing = [n for n in need if n not in have]
print("missing nodes:", missing if missing else "none")
if missing:
    proc.terminate()
    raise SystemExit("missing nodes: " + ", ".join(missing))

def node_inputs(class_type, preferred):
    spec = info[class_type]["input"]
    allowed = set(spec.get("required", {})) | set(spec.get("optional", { }))
    return {k: v for k, v in preferred.items() if k in allowed}

# Print the loader + sampler + clip specs to the log so a wrong input name/value is caught here, not after a long run.
for ct in ["UNETLoader", "CLIPLoader", "ModelSamplingAuraFlow", "KSampler", "EmptySD3LatentImage"]:
    print(ct, "spec:", json.dumps(info[ct]["input"])[:1200])

# Raw-node version of the official Z-Image Turbo T2I template (subgraph expanded,
# prompt-enhancement branch dropped, prompt sent as-is). Core pipeline:
# UNETLoader -> ModelSamplingAuraFlow (shift 3) -> KSampler 8 steps, cfg 1,
# res_multistep, simple; CLIPLoader type lumina2 -> CLIPTextEncode ->
# ConditioningZeroOut (negative); EmptySD3LatentImage 1024x1024; VAEDecode; SaveImage.
nodes = {}
nodes["1"] = {"class_type": "UNETLoader", "inputs": node_inputs("UNETLoader", {"unet_name": "z_image_turbo_int8_convrot.safetensors", "weight_dtype": "default"})}
nodes["2"] = {"class_type": "CLIPLoader", "inputs": node_inputs("CLIPLoader", {"clip_name": "qwen_3_4b_fp8_mixed.safetensors", "type": "lumina2", "device": "default"})}
nodes["3"] = {"class_type": "VAELoader", "inputs": node_inputs("VAELoader", {"vae_name": "ae.safetensors"})}
nodes["4"] = {"class_type": "CLIPTextEncode", "inputs": {"text": PROMPT, "clip": ["2", 0]}}
nodes["5"] = {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["4", 0]}}
nodes["6"] = {"class_type": "EmptySD3LatentImage", "inputs": node_inputs("EmptySD3LatentImage", {"width": WIDTH, "height": HEIGHT, "batch_size": 1})}
# Model line: UNETLoader -> (LoRA chain) -> ModelSamplingAuraFlow (shift 3) -> KSampler.
# LoRA chain: one LoraLoaderModelOnly per entry of LORA_NAMES, in order (model-only,
# no CLIP). Empty list / no LORA_NAME = model goes straight to ModelSamplingAuraFlow.
MODEL_OUT = ["1", 0]
_lora_chain = LORA_NAMES
if _lora_chain is None and LORA_NAME:
    _lora_chain = [(LORA_NAME, LORA_STRENGTH, "")]
if _lora_chain:
    for _i, (_lname, _lstr, _lds) in enumerate(_lora_chain):
        _nid = str(20 + _i)
        nodes[_nid] = {"class_type": "LoraLoaderModelOnly", "inputs": node_inputs("LoraLoaderModelOnly", {"model": MODEL_OUT, "lora_name": _lname, "strength_model": _lstr})}
        MODEL_OUT = [_nid, 0]
nodes["7"] = {"class_type": "ModelSamplingAuraFlow", "inputs": node_inputs("ModelSamplingAuraFlow", {"model": MODEL_OUT, "shift": 3})}
nodes["8"] = {"class_type": "KSampler", "inputs": {"model": ["7", 0], "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0], "seed": 42, "steps": 8, "cfg": 1, "sampler_name": "res_multistep", "scheduler": "simple", "denoise": 1.0}}
nodes["9"] = {"class_type": "VAEDecode", "inputs": {"samples": ["8", 0], "vae": ["3", 0]}}
nodes["10"] = {"class_type": "SaveImage", "inputs": {"images": ["9", 0], "filename_prefix": "zimage_lowres"}}

# Pre-check: every required input of every node must be present, or fail here with the
# full list (do not let the server reject it one input at a time). Plus a combo-value
# check: where the node definition lists the allowed values, the value we send must be
# one of them (a bad value can be silently dropped and only fail at execution -- the Wan
# V5 lesson). NOTE: a combo's allowed values live in cfg[0] as a LIST; a scalar input
# (STRING/INT/FLOAT) has cfg[0] as a type-name string, so it is skipped -- this is the
# exact check the Qwen/Krea2 notebooks use.
problems = []
for nid, node in nodes.items():
    spec = info[node["class_type"]]["input"]
    req = spec.get("required", {})
    for name in req:
        if name not in node["inputs"]:
            problems.append(nid + ":" + node["class_type"] + "." + name)
    for name, val in node["inputs"].items():
        cfg = req.get(name) or spec.get("optional", {}).get(name)
        if isinstance(cfg, list) and cfg and isinstance(cfg[0], list) and isinstance(val, (str, int, float)):
            if val not in cfg[0]:
                problems.append(nid + ":" + node["class_type"] + "." + name + " value '" + str(val) + "' not in " + str(cfg[0]))
print("missing required inputs:", problems if problems else "none")
if problems:
    proc.terminate()
    raise SystemExit("missing required inputs: " + ", ".join(problems))

# ---------- SeedVR helpers (from the verified 7B test notebook) ----------
def combo_opts_for_cfg(cfg):
    """Options list for a combo input spec, or None if the input is not a combo.
    Old format: [["opt1", "opt2"], {...}]. New format (current ComfyUI master /
    SeedVR2 plugin): ["COMBO", {"options": [...], "default": ..., ...}]."""
    if not isinstance(cfg, list) or not cfg:
        return None
    if isinstance(cfg[0], list):
        return cfg[0]
    if cfg[0] == "COMBO" and len(cfg) > 1 and isinstance(cfg[1], dict) and "options" in cfg[1]:
        return cfg[1]["options"]
    return None

def combo_options(class_type, name):
    """Return the option list for a combo input (both spec formats)."""
    spec = info[class_type]["input"]
    cfg = spec.get("required", {}).get(name) or spec.get("optional", {}).get(name)
    opts = combo_opts_for_cfg(cfg)
    return opts if opts is not None else ([cfg[0]] if cfg else [])

# Print the SeedVR2 node specs so a wrong input name/value is caught here, not after a long run.
for ct in ["SeedVR2LoadDiTModel", "SeedVR2LoadVAEModel", "SeedVR2VideoUpscaler"]:
    print(ct, "spec:", json.dumps(info[ct]["input"])[:1200])

# Resolve device + model values from the live specs (never hard-code a combo value).
dit_device = combo_options("SeedVR2LoadDiTModel", "device")[0]
vae_device = combo_options("SeedVR2LoadVAEModel", "device")[0]
dit_models = combo_options("SeedVR2LoadDiTModel", "model")
vae_models = combo_options("SeedVR2LoadVAEModel", "model")
print("dit device:", dit_device, "| vae device:", vae_device)
print("dit models available:", dit_models)
print("vae models available:", vae_models)
DIT_MODEL = "seedvr2_ema_7b_fp8_e4m3fn_mixed_block35_fp16.safetensors"
VAE_MODEL = "ema_vae_fp16.safetensors"
assert DIT_MODEL in dit_models, "DiT model not in loader options: " + DIT_MODEL
assert VAE_MODEL in vae_models, "VAE model not in loader options: " + VAE_MODEL

# ---------- Phase 2 nodes: SeedVR2 7B upscale (1024x1536 -> 2048x3072) ----------
# Verified 2026-11-15 (kernel kitzhang2019/seedvr-7b-ddimbeta-test-20261114):
# resolution 2048, max 6000, batch 5, blocks_to_swap 36, VAE tiled 1024/128,
# sdpa attention, color lab. 390 s on T4; 4096 OOMs in DiT attention.
nodes_seedvr = {}
nodes_seedvr["31"] = {"class_type": "LoadImage", "inputs": node_inputs("LoadImage", {"image": "zimage_lowres.png"})}
nodes_seedvr["32"] = {"class_type": "SeedVR2LoadDiTModel", "inputs": node_inputs("SeedVR2LoadDiTModel", {
    "model": DIT_MODEL, "device": dit_device, "blocks_to_swap": 36,
    "swap_io_components": False, "offload_device": "cpu", "attention_mode": "sdpa"})}
nodes_seedvr["33"] = {"class_type": "SeedVR2LoadVAEModel", "inputs": node_inputs("SeedVR2LoadVAEModel", {
    "model": VAE_MODEL, "device": vae_device, "encode_tiled": True, "encode_tile_size": 1024,
    "encode_tile_overlap": 128, "decode_tiled": True, "decode_tile_size": 1024,
    "decode_tile_overlap": 128, "tile_debug": "false", "offload_device": "cpu"})}
nodes_seedvr["34"] = {"class_type": "SeedVR2VideoUpscaler", "inputs": node_inputs("SeedVR2VideoUpscaler", {
    "image": ["31", 0], "dit": ["32", 0], "vae": ["33", 0], "seed": 42,
    "resolution": 2048, "max_resolution": 6000, "batch_size": 5,
    "uniform_batch_size": False, "temporal_overlap": 0, "prepend_frames": 0,
    "color_correction": "lab", "input_noise_scale": 0.0})}
nodes_seedvr["35"] = {"class_type": "SaveImage", "inputs": {"images": ["34", 0], "filename_prefix": "zimage_test"}}


def submit_and_wait(nodes, save_node_id, out_path, timeout_s, label):
    """Submit one prompt, poll until success/error/timeout, download the SaveImage output. Returns (done, elapsed_s)."""
    r = requests.post(BASE + "/prompt", json={"prompt": nodes}, timeout=180)
    print(label, "submit:", r.status_code, r.text[:400])
    r.raise_for_status()
    pid = r.json()["prompt_id"]
    print(label, "prompt_id:", pid)
    t0 = time.time()
    done = False
    while time.time() - t0 < timeout_s:
        h = requests.get(BASE + "/history/" + pid, timeout=60).json()
        if pid in h:
            st = h[pid].get("status", {})
            sstr = st.get("status_str")
            if sstr == "success":
                outs = h[pid]["outputs"]
                imgs = outs.get(save_node_id, {}).get("images", [])
                print(label, "images:", imgs)
                if imgs:
                    im = imgs[0]
                    q = {"filename": im["filename"], "subfolder": im.get("subfolder", ""), "type": im.get("type", "output")}
                    data = requests.get(BASE + "/view", params=q, timeout=300).content
                    open(out_path, "wb").write(data)
                    print(label, "saved", out_path, len(data), "bytes")
                    done = True
                break
            if sstr == "error":
                print(label, "RUN ERROR:", json.dumps(h[pid], indent=2)[:3000])
                break
        time.sleep(15)
    elapsed = round(time.time() - t0)
    print(label, "done:", done, "elapsed:", elapsed, "s")
    return done, elapsed

# Phase 1: low-res generation (1024x1536). Z-Image 8 steps, expect a few min; allow 60 min.
gen_ok, gen_s = submit_and_wait(nodes, "10", "/kaggle/working/zimage_lowres.png", 3600, "GEN")
if not gen_ok:
    proc.terminate()
    raise SystemExit("phase 1 (generation) failed")

# LoadImage reads from ComfyUI/input/; copy the low-res output there.
import shutil
os.makedirs("/kaggle/working/ComfyUI/input", exist_ok=True)
shutil.copyfile("/kaggle/working/zimage_lowres.png", "/kaggle/working/ComfyUI/input/zimage_lowres.png")
print("copied low-res to ComfyUI/input/zimage_lowres.png")

# Re-fetch object_info now that the low-res file is in ComfyUI/input/:
# the LoadImage combo options are snapshotted from the input dir at fetch
# time, so the pre-check must see the file Phase 1 just produced.
info = requests.get(BASE + "/object_info", timeout=120).json()

# Pre-check for the SeedVR nodes: every required input present, and every
# combo value one of the live options (same check as the generation phase).
problems_sv = []
for nid, node in nodes_seedvr.items():
    spec = info[node["class_type"]]["input"]
    req = spec.get("required", {})
    for name in req:
        if name not in node["inputs"]:
            problems_sv.append(nid + ":" + node["class_type"] + "." + name)
    for name, val in node["inputs"].items():
        cfg = req.get(name) or spec.get("optional", {}).get(name)
        opts = combo_opts_for_cfg(cfg)
        if opts is not None and isinstance(val, (str, int, float)):
            if val not in opts:
                problems_sv.append(nid + ":" + node["class_type"] + "." + name + " value '" + str(val) + "' not in " + str(opts))
print("seedvr missing required inputs:", problems_sv if problems_sv else "none")
if problems_sv:
    proc.terminate()
    raise SystemExit("seedvr missing required inputs: " + ", ".join(problems_sv))

# Phase 2: SeedVR2 7B upscale (1024x1536 -> 2048x3072). Verified ~390 s; allow 30 min.
sv_ok, sv_s = submit_and_wait(nodes_seedvr, "35", "/kaggle/working/zimage_test.png", 1800, "SEEDVR")
if not sv_ok:
    proc.terminate()
    raise SystemExit("phase 2 (SeedVR upscale) failed")

# Phase 3: WebP fine export (quality 95 / method 6; step down to 90, then 85 only
# if still over 20MB). The 2048x3072 PNG is ~8.8MB, over the station's 8MB
# return cap, so the WebP is the file the station fetches back.
from PIL import Image as _WImage
def webp_fine(png_path, webp_path):
    sz = None
    q = None
    for _q in (95, 90, 85):
        _WImage.open(png_path).convert("RGB").save(webp_path, "WEBP", quality=_q, method=6)
        _sz = os.path.getsize(webp_path)
        print("webp q%d" % _q, webp_path, _sz, "bytes")
        sz, q = _sz, _q
        if _sz <= 20 * 1024 * 1024 or _q == 85:
            break
    return sz, q

webp_s, webp_q = webp_fine("/kaggle/working/zimage_test.png", "/kaggle/working/zimage_test.webp")

proc.terminate()
print("server stopped")
print("RESULT gen_s=%d seedvr_s=%d webp_bytes=%d webp_q=%d" % (gen_s, sv_s, webp_s, webp_q))
print("exists:", os.path.exists("/kaggle/working/zimage_test.webp"), "bytes:", os.path.getsize("/kaggle/working/zimage_test.webp") if os.path.exists("/kaggle/working/zimage_test.webp") else 0)
assert os.path.exists("/kaggle/working/zimage_test.webp"), "no output webp: /kaggle/working/zimage_test.webp"


In [ ]:
from IPython.display import Image, display
import os
# NOTE: do NOT display the .webp here -- IPython cannot embed webp
# ("Cannot embed the 'webp' image format") and papermill marks the whole
# kernel ERROR even though generation succeeded. Only the PNG is displayed.
for p in ["/kaggle/working/zimage_test.png", "/kaggle/working/zimage_test.webp"]:
    print(p, "exists:", os.path.exists(p), "bytes:", os.path.getsize(p) if os.path.exists(p) else 0)
if os.path.exists("/kaggle/working/zimage_test.png"):
    display(Image(filename="/kaggle/working/zimage_test.png"))
